In [35]:
'''
Step 1 SQL: Load all_stocks_5yr.csv into PostgreSQL via psycopg2.
Create a stocks table with consistent (ticker, date) keys.
The dataset contains ~500 S&P 500 stocks with daily OHLCV data from 2013 to 2018.
'''
import pandas as pd
import psycopg2
import os

from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT # to create databases
from psycopg2.extras import execute_batch

from typing import Iterable, Callable

db_nm = 'postgres'
usr_nm = 'postgres'
host = 'localhost'
port = '5432'

path_to_pwd_file = r"C:\Users\timur\Desktop\Python_projects_2026\postgresql password.txt"

with open(path_to_pwd_file, 'r') as file:
    pwd = file.read().strip('\n')

def connect(pwd: str,
            usr_nm: str = 'postgres', 
            host: str = 'localhost',
            port: str = '5432',
            db_nm: str = 'postgres'
           ) -> psycopg2.extensions.connection | None:
    conn = None
    try:
        conn = psycopg2.connect(
            host=host,
            database=db_nm,
            user=usr_nm,
            port=port,
            password=pwd
        )
    except:
        print('Connection failure!')
    else:
        print('Connection success!')
    finally:
        return conn
    
def create_db(new_db_nm: str,
              conn: psycopg2.extensions.connection | None = None,
              cur: psycopg2.extensions.cursor | None = None
             ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor]:
    
    # if no connection passed, create one to the default postgres db
    if conn is None:
        conn = connect(pwd, db_nm='postgres')
    
    # autocommit is required for CREATE DATABASE — it can't run inside a transaction
    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    
    if cur is None:
        cur = conn.cursor()
    
    try:
        cur.execute(f"CREATE DATABASE {new_db_nm};")
        print(f"Database '{new_db_nm}' created successfully!")
    except psycopg2.errors.DuplicateDatabase:
        print(f"Database '{new_db_nm}' already exists, skipping creation.")
    
    # reset isolation level back to default
    conn.set_isolation_level(0)
    
    return conn, cur

conn, cur = create_db('stocks_db')

cur = conn.cursor()

# read the CSV with pandas
df = pd.read_csv(r'C:\Users\timur\Desktop\Python_projects_2026\archive\all_stocks_5yr.csv')
df['date'] = pd.to_datetime(df['date'])

# create the table first
cur.execute("""
    CREATE TABLE IF NOT EXISTS stocks (
        date    DATE,
        open    FLOAT,
        high    FLOAT,
        low     FLOAT,
        close   FLOAT,
        volume  BIGINT,
        ticker  VARCHAR(10)
    );
""")
conn.commit()

# insert in batches of 1000 rows using execute_batch (much faster than executemany)
records = list(df.itertuples(index=False, name=None))
execute_batch(cur, """
    INSERT INTO stocks (date, open, high, low, close, volume, ticker)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", records, page_size=1000)

conn.commit()
print(f"Loaded {len(df)} rows successfully!")

Connection success!
Database 'stocks_db' already exists, skipping creation.
Loaded 619040 rows successfully!


## Step 1 — Data Import

We load `all_stocks_5yr.csv` into a PostgreSQL database using psycopg2. The dataset contains daily OHLCV (Open, High, Low, Close, Volume) price data for ~500 S&P 500 stocks from 2013 to 2018, with a `ticker` column identifying each stock.

We chose PostgreSQL over pure pandas for this project because SQL window functions (`LAG`, `STDDEV`, `AVG OVER`) are purpose-built for time-series calculations like rolling stats and returns, and a persistent database means we don't reload the CSV on every run.

**Why `execute_batch` instead of `COPY FROM`?** PostgreSQL's `COPY FROM` runs as the server process and can't access files in user directories — it threw a `Permission Denied` error. Instead we read the CSV with pandas and insert via `execute_batch`, which runs client-side and sends data over the connection.

**Why `ISOLATION_LEVEL_AUTOCOMMIT` in `create_db`?** `CREATE DATABASE` cannot run inside a transaction block, which is how psycopg2 operates by default. We set autocommit mode for that one command, then reset it back to 0 so all subsequent operations behave normally.

In [36]:
'''
Step 2 SQL: Calculate daily
returns and log returns for each stock using LAG() window functions,
partitioned by ticker and ordered by date.
'''
def calculate_returns(conn: psycopg2.extensions.connection,
                      cur: psycopg2.extensions.cursor) -> None:
    cur.execute("""
        ALTER TABLE stocks 
        ADD COLUMN IF NOT EXISTS daily_return FLOAT,
        ADD COLUMN IF NOT EXISTS log_return FLOAT;
        
        UPDATE stocks s
        SET 
            daily_return = (s.close - prev.prev_close) / prev.prev_close,
            log_return   = LN(s.close / prev.prev_close)
        FROM (
            SELECT ticker, date,
                   LAG(close) OVER (PARTITION BY ticker ORDER BY date) AS prev_close
            FROM stocks
        ) prev
        WHERE s.ticker = prev.ticker 
          AND s.date   = prev.date
          AND prev.prev_close IS NOT NULL;
    """)
    conn.commit()
    print('Returns calculated successfully!')

calculate_returns(conn, cur)

Returns calculated successfully!


## Step 2 — Daily & Log Returns

We calculate two return metrics for each stock using SQL window functions:

- **Daily return:** `(today_close - yesterday_close) / yesterday_close` — the standard percentage change, intuitive and easy to interpret.
- **Log return:** `LN(today_close / yesterday_close)` — the natural log of the price ratio. Log returns are additive over time (you can sum them across a period without compounding adjustments) and are more normally distributed, which matters for the statistical calculations in Steps 3 and 4.

The key SQL technique is `LAG(close) OVER (PARTITION BY ticker ORDER BY date)`:
- `LAG(close)` retrieves the previous row's closing price — equivalent to `iloc[i-1]` in pandas.
- `PARTITION BY ticker` ensures the window resets for each stock, so we never accidentally use the last row of one ticker as "yesterday" for the next ticker.
- The first row of each ticker gets `NULL` since there's no previous price, which we handle with the `IS NOT NULL` filter in the `WHERE` clause.

In [37]:
'''
Step 3 SQL: Rolling statistics over
7, 14, 30, 60, 90-day windows mean return, std dev, min, max.
Replaces your incomplete moving_average_func().
'''
def calculate_rolling_stats(conn: psycopg2.extensions.connection,
                             cur: psycopg2.extensions.cursor) -> None:
    cur.execute("""
        ALTER TABLE stocks
        ADD COLUMN IF NOT EXISTS rolling_mean_7   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_std_7    FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_min_7    FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_max_7    FLOAT,

        ADD COLUMN IF NOT EXISTS rolling_mean_14  FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_std_14   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_min_14   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_max_14   FLOAT,

        ADD COLUMN IF NOT EXISTS rolling_mean_30  FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_std_30   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_min_30   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_max_30   FLOAT,

        ADD COLUMN IF NOT EXISTS rolling_mean_60  FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_std_60   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_min_60   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_max_60   FLOAT,

        ADD COLUMN IF NOT EXISTS rolling_mean_90  FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_std_90   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_min_90   FLOAT,
        ADD COLUMN IF NOT EXISTS rolling_max_90   FLOAT;

        UPDATE stocks s
        SET
            -- 7-day
            rolling_mean_7  = r.mean_7,
            rolling_std_7   = r.std_7,
            rolling_min_7   = r.min_7,
            rolling_max_7   = r.max_7,
            -- 14-day
            rolling_mean_14 = r.mean_14,
            rolling_std_14  = r.std_14,
            rolling_min_14  = r.min_14,
            rolling_max_14  = r.max_14,
            -- 30-day
            rolling_mean_30 = r.mean_30,
            rolling_std_30  = r.std_30,
            rolling_min_30  = r.min_30,
            rolling_max_30  = r.max_30,
            -- 60-day
            rolling_mean_60 = r.mean_60,
            rolling_std_60  = r.std_60,
            rolling_min_60  = r.min_60,
            rolling_max_60  = r.max_60,
            -- 90-day
            rolling_mean_90 = r.mean_90,
            rolling_std_90  = r.std_90,
            rolling_min_90  = r.min_90,
            rolling_max_90  = r.max_90
        FROM (
            SELECT ticker, date,
                -- 7-day window
                AVG(daily_return)  OVER w7 AS mean_7,
                STDDEV(daily_return) OVER w7 AS std_7,
                MIN(daily_return)  OVER w7 AS min_7,
                MAX(daily_return)  OVER w7 AS max_7,
                -- 14-day window
                AVG(daily_return)  OVER w14 AS mean_14,
                STDDEV(daily_return) OVER w14 AS std_14,
                MIN(daily_return)  OVER w14 AS min_14,
                MAX(daily_return)  OVER w14 AS max_14,
                -- 30-day window
                AVG(daily_return)  OVER w30 AS mean_30,
                STDDEV(daily_return) OVER w30 AS std_30,
                MIN(daily_return)  OVER w30 AS min_30,
                MAX(daily_return)  OVER w30 AS max_30,
                -- 60-day window
                AVG(daily_return)  OVER w60 AS mean_60,
                STDDEV(daily_return) OVER w60 AS std_60,
                MIN(daily_return)  OVER w60 AS min_60,
                MAX(daily_return)  OVER w60 AS max_60,
                -- 90-day window
                AVG(daily_return)  OVER w90 AS mean_90,
                STDDEV(daily_return) OVER w90 AS std_90,
                MIN(daily_return)  OVER w90 AS min_90,
                MAX(daily_return)  OVER w90 AS max_90
            FROM stocks
            WINDOW
                w7  AS (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6  PRECEDING AND CURRENT ROW),
                w14 AS (PARTITION BY ticker ORDER BY date ROWS BETWEEN 13 PRECEDING AND CURRENT ROW),
                w30 AS (PARTITION BY ticker ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                w60 AS (PARTITION BY ticker ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                w90 AS (PARTITION BY ticker ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW)
        ) r
        WHERE s.ticker = r.ticker
          AND s.date   = r.date;
    """)
    conn.commit()
    print('Rolling statistics calculated successfully!')

calculate_rolling_stats(conn, cur)

Rolling statistics calculated successfully!


## Step 3 — Rolling Statistics

For each stock and each trading day we calculate rolling mean, standard deviation, minimum and maximum of **daily returns** over 5 window sizes: 7, 14, 30, 60, and 90 days.

We compute these on `daily_return` values (not raw prices) because rolling std of returns equals volatility, which feeds directly into Step 4's regime classification.

The SQL technique used is named `WINDOW` definitions:
```sql
WINDOW w7 AS (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)
```
- `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` gives exactly 7 rows (6 previous + current). For an N-day window the formula is always `N-1 PRECEDING`.
- Named windows avoid repeating the full definition 20 times across all metrics.
- For early rows where fewer than N previous rows exist, PostgreSQL uses whatever rows are available — standard behavior for rolling calculations.

In [38]:
'''
Step 4 SQL: Classify each day into a volatility regime (Low / Normal / High / Extreme)
based on the 30-day rolling std dev. Track how long each stock stays in each regime.
'''
def calculate_volatility_regimes(conn: psycopg2.extensions.connection,
                                  cur: psycopg2.extensions.cursor) -> None:
    cur.execute("""
        ALTER TABLE stocks
        ADD COLUMN IF NOT EXISTS volatility_regime VARCHAR(10);

        UPDATE stocks s
        SET volatility_regime = CASE
            WHEN r.rolling_std_30 < 0.01   THEN 'Low'
            WHEN r.rolling_std_30 < 0.02   THEN 'Normal'
            WHEN r.rolling_std_30 < 0.035  THEN 'High'
            ELSE                                 'Extreme'
        END
        FROM stocks r
        WHERE s.ticker = r.ticker
          AND s.date   = r.date
          AND r.rolling_std_30 IS NOT NULL;
    """)
    conn.commit()
    print('Volatility regimes classified successfully!')

calculate_volatility_regimes(conn, cur)

Volatility regimes classified successfully!


## Step 4 — Volatility Regime Classification

Each trading day is classified into one of four volatility regimes based on the 30-day rolling standard deviation of returns:

| Regime  | Threshold |
|---------|-----------|
| Low     | `rolling_std_30 < 1%` |
| Normal  | `1% ≤ rolling_std_30 < 2%` |
| High    | `2% ≤ rolling_std_30 < 3.5%` |
| Extreme | `rolling_std_30 ≥ 3.5%` |

These thresholds are in the same units as `daily_return` — decimal percentages. So `0.02` means the stock's daily returns have been swinging by ~2% on average over the last 30 days.

**Why `rolling_std_30` and not 7 or 90?** A 7-day window is too noisy — stocks would flip between regimes almost daily. A 90-day window is too slow to react to real volatility spikes. 30 days (~1 trading month) balances stability with reactivity, consistent with standard financial volatility indicators.

The classification uses a SQL `CASE` statement, equivalent to a Python `if/elif/else` chain.

In [39]:
'''
Step 5 Python/pandas: Cumulative returns over user-defined date ranges,
annualized for comparison. Extends your existing cumulative_return_func().
'''
def calculate_cumulative_returns(conn: psycopg2.extensions.connection,
                                  cur: psycopg2.extensions.cursor,
                                  start_date: str,
                                  end_date: str) -> pd.DataFrame:
    cur.execute("""
        SELECT ticker,
               EXP(SUM(LN(1 + daily_return))) - 1                    AS cumulative_return,
               POWER(EXP(SUM(LN(1 + daily_return))), 252.0 / COUNT(*)) - 1  AS annualized_return
        FROM stocks
        WHERE date BETWEEN %s AND %s
          AND daily_return IS NOT NULL
        GROUP BY ticker
        ORDER BY annualized_return DESC;
    """, (start_date, end_date))
    
    rows = cur.fetchall()
    df = pd.DataFrame(rows, columns=['ticker', 'cumulative_return', 'annualized_return'])
    print(df)
    return df

calculate_cumulative_returns(conn, cur, '2007-03-05', '2016-08-30')

    ticker  cumulative_return  annualized_return
0      STZ          26.213338           0.590528
1     AVGO          24.113379           0.572687
2     NVDA          23.822467           0.570115
3       EA          21.139273           0.545086
4     INCY          18.539102           0.518207
..     ...                ...                ...
493  DISCK          -0.852382          -0.235653
494   CSRA          -0.358996          -0.245411
495  DISCA          -0.871799          -0.250646
496    CHK          -0.896445          -0.272785
497    FCX          -0.912209          -0.289460

[498 rows x 3 columns]


,ticker,cumulative_return,annualized_return
0,STZ,26.213338,0.590528
1,AVGO,24.113379,0.572687
2,NVDA,23.822467,0.570115
3,EA,21.139273,0.545086
4,INCY,18.539102,0.518207
...,...,...,...
493,DISCK,-0.852382,-0.235653
494,CSRA,-0.358996,-0.245411
495,DISCA,-0.871799,-0.250646
496,CHK,-0.896445,-0.272785


## Step 5 — Cumulative & Annualized Returns

For a user-defined date range we calculate two return metrics per stock:

- **Cumulative return:** `EXP(SUM(LN(1 + daily_return))) - 1`
- **Annualized return:** `POWER(cumulative + 1, 252 / trading_days) - 1`

**Why not `SUM(daily_return)`?** Returns compound — a 10% gain followed by a 10% gain is 21%, not 20%, because the second gain applies to a larger base. `SUM` ignores this. We use the log trick instead: `LN` converts multiplication into addition, `SUM` adds them all up, and `EXP` converts back. The result is identical to multiplying all `(1 + r)` terms together, but SQL doesn't have a built-in product aggregation function.

**Why 252?** That is the number of trading days in a year — markets are closed on weekends and public holidays. Annualizing allows fair comparison between periods of different lengths.

In [40]:
'''
Step 6 Python/pandas: Risk metrics Sharpe ratio, max drawdown, Calmar ratio.
Aggregated at sector level too.
'''
def calculate_risk_metrics(conn: psycopg2.extensions.connection,
                            cur: psycopg2.extensions.cursor) -> pd.DataFrame:
    cur.execute("""
        WITH daily AS (
            SELECT ticker,
                   date,
                   daily_return,
                   -- running cumulative return to track equity curve
                   EXP(SUM(LN(1 + daily_return)) 
                       OVER (PARTITION BY ticker ORDER BY date)) AS equity_curve
            FROM stocks
            WHERE daily_return IS NOT NULL
        ),
        drawdown AS (
            SELECT ticker,
                   date,
                   daily_return,
                   equity_curve,
                   -- peak so far for this ticker
                   MAX(equity_curve) OVER (PARTITION BY ticker ORDER BY date) AS peak,
                   -- drawdown at each point
                   (equity_curve - MAX(equity_curve) 
                       OVER (PARTITION BY ticker ORDER BY date)) 
                   / MAX(equity_curve) 
                       OVER (PARTITION BY ticker ORDER BY date)       AS drawdown
            FROM daily
        )
        SELECT 
            ticker,
            -- annualized return
            (POWER(MAX(equity_curve), 252.0 / COUNT(*)) - 1)          AS annualized_return,
            -- annualized volatility
            STDDEV(daily_return) * SQRT(252)                           AS annualized_volatility,
            -- sharpe ratio (assuming 0 risk-free rate for simplicity)
            (AVG(daily_return) * 252) / (STDDEV(daily_return) * SQRT(252)) AS sharpe_ratio,
            -- max drawdown
            MIN(drawdown)                                              AS max_drawdown,
            -- calmar ratio
            (POWER(MAX(equity_curve), 252.0 / COUNT(*)) - 1) 
            / ABS(MIN(drawdown))                                       AS calmar_ratio
        FROM drawdown
        GROUP BY ticker
        ORDER BY sharpe_ratio DESC;
    """)
    
    rows = cur.fetchall()
    df = pd.DataFrame(rows, columns=[
        'ticker', 'annualized_return', 'annualized_volatility',
        'sharpe_ratio', 'max_drawdown', 'calmar_ratio'
    ])
    print(df)
    return df

calculate_risk_metrics(conn, cur)

    ticker  annualized_return  annualized_volatility  sharpe_ratio  \
0      DXC           0.619045               0.194174      2.188542   
1      HLT           0.458517               0.163732      2.015241   
2      NOC           0.390304               0.173049      1.973894   
3     NVDA           0.820620               0.354593      1.819767   
4      LMT           0.323782               0.160260      1.788580   
..     ...                ...                    ...           ...   
500    RRC           0.053704               0.435926     -0.564659   
501    BHF           0.000000               0.273125     -0.964528   
502   EVHC           0.031106               0.452248     -1.047708   
503     UA           0.053302               0.453208     -1.233567   
504   BHGE           0.000000               0.556331     -1.727360   

     max_drawdown  calmar_ratio  
0       -0.154577      4.004770  
1       -0.185963      2.465636  
2       -0.207673      1.879420  
3       -0.441604      

,ticker,annualized_return,annualized_volatility,sharpe_ratio,max_drawdown,calmar_ratio
0,DXC,0.619045,0.194174,2.188542,-0.154577,4.004770
1,HLT,0.458517,0.163732,2.015241,-0.185963,2.465636
2,NOC,0.390304,0.173049,1.973894,-0.207673,1.879420
3,NVDA,0.820620,0.354593,1.819767,-0.441604,1.858272
4,LMT,0.323782,0.160260,1.788580,-0.251791,1.285913
...,...,...,...,...,...,...
500,RRC,0.053704,0.435926,-0.564659,-0.980454,0.054775
501,BHF,0.000000,0.273125,-0.964528,-0.426735,0.000000
502,EVHC,0.031106,0.452248,-1.047708,-0.883019,0.035227
503,UA,0.053302,0.453208,-1.233567,-0.946372,0.056322


## Step 6 — Risk Metrics (Sharpe, Max Drawdown, Calmar)

We calculate three risk-adjusted performance metrics per stock:

- **Sharpe Ratio:** `annualized_return / annualized_volatility` — measures return per unit of risk. A Sharpe of 2.0 means the stock earned 2% of return for every 1% of volatility. We assume a risk-free rate of 0 for simplicity; this slightly overstates all Sharpe ratios equally so relative comparisons remain valid.
- **Max Drawdown:** the largest peak-to-trough decline in the equity curve. Calculated by tracking the running peak and measuring how far below it the stock falls at each point. Represents the worst loss an investor would have experienced if they bought at the peak.
- **Calmar Ratio:** `annualized_return / |max_drawdown|` — similar to Sharpe but uses drawdown as the risk measure instead of volatility.

**Why `STDDEV * SQRT(252)`?** Variance scales linearly with time but standard deviation scales with the square root, so `daily_std × √252` correctly converts daily volatility to annual volatility.

The equity curve is built using `EXP(SUM(LN(1 + daily_return)) OVER (...))` — the same compounding trick as Step 5, applied as a running window function.

In [ ]:
'''
Step 7 SQL: Build the final consolidated table one row per stock per day with everything:
prices, returns, rolling stats, regime label, risk metrics. Indexed on (ticker, date).
'''
def build_final_table(conn: psycopg2.extensions.connection,
                      cur: psycopg2.extensions.cursor) -> None:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS stocks_final AS
        SELECT
            s.ticker,
            s.date,
            s.open,
            s.high,
            s.low,
            s.close,
            s.volume,
            -- returns
            s.daily_return,
            s.log_return,
            -- rolling stats
            s.rolling_mean_7,   s.rolling_std_7,   s.rolling_min_7,   s.rolling_max_7,
            s.rolling_mean_14,  s.rolling_std_14,  s.rolling_min_14,  s.rolling_max_14,
            s.rolling_mean_30,  s.rolling_std_30,  s.rolling_min_30,  s.rolling_max_30,
            s.rolling_mean_60,  s.rolling_std_60,  s.rolling_min_60,  s.rolling_max_60,
            s.rolling_mean_90,  s.rolling_std_90,  s.rolling_min_90,  s.rolling_max_90,
            -- regime
            s.volatility_regime,
            -- risk metrics joined in
            r.annualized_return,
            r.annualized_volatility,
            r.sharpe_ratio,
            r.max_drawdown,
            r.calmar_ratio
        FROM stocks s
        LEFT JOIN (
            WITH daily AS (
                SELECT ticker, date, daily_return,
                       EXP(SUM(LN(1 + daily_return)) 
                           OVER (PARTITION BY ticker ORDER BY date)) AS equity_curve
                FROM stocks WHERE daily_return IS NOT NULL
            ),
            drawdown AS (
                SELECT ticker, date, daily_return, equity_curve,
                       MAX(equity_curve) OVER (PARTITION BY ticker ORDER BY date) AS peak,
                       (equity_curve - MAX(equity_curve) OVER (PARTITION BY ticker ORDER BY date))
                       / MAX(equity_curve) OVER (PARTITION BY ticker ORDER BY date) AS drawdown
                FROM daily
            )
            SELECT ticker,
                   (POWER(MAX(equity_curve), 252.0 / COUNT(*)) - 1)              AS annualized_return,
                   STDDEV(daily_return) * SQRT(252)                               AS annualized_volatility,
                   (AVG(daily_return) * 252) / (STDDEV(daily_return) * SQRT(252)) AS sharpe_ratio,
                   MIN(drawdown)                                                   AS max_drawdown,
                   (POWER(MAX(equity_curve), 252.0 / COUNT(*)) - 1) 
                   / ABS(MIN(drawdown))                                           AS calmar_ratio
            FROM drawdown
            GROUP BY ticker
        ) r ON s.ticker = r.ticker
        ORDER BY s.ticker, s.date;

        CREATE INDEX IF NOT EXISTS idx_final_ticker_date 
        ON stocks_final (ticker, date);
    """)
    conn.commit()
    print('Final table built successfully!')

build_final_table(conn,cur)

Final table built successfully!


## Step 7 — Final Consolidated Table

We build a single `stocks_final` table containing one row per stock per day with every calculated metric: raw prices, daily and log returns, all rolling statistics across 5 windows, volatility regime classification, and risk metrics.

**Why a consolidated table instead of querying `stocks` directly each time?**
- **Performance:** all joins and window calculations are done once and stored. Subsequent queries on `stocks_final` are simple `SELECT`s with no heavy computation.
- **Cleanliness:** analysis and visualization cells only need to read from one place, keeping the notebook readable.
- **Indexing:** we add a `(ticker, date)` index which makes filtering by stock or date range very fast.

Risk metrics are attached via a `LEFT JOIN` rather than `INNER JOIN` — if a ticker had insufficient data to calculate risk metrics, its price and return rows still appear in the final table with `NULL` risk columns rather than being silently dropped.

We also export the final table to `stocks_final.csv` using `cur.description` to auto-extract column names from PostgreSQL — this avoids hardcoding the 35 column names manually.

In [54]:
def export_final_table(conn, cur, export_path: str) -> pd.DataFrame:
    cur.execute("SELECT * FROM stocks_final;")
    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]
    df = pd.DataFrame(rows, columns=columns)
    df.to_csv(export_path, index=False)
    print(f'Exported {len(df)} rows to {export_path}')
    return df
cur.execute("DROP TABLE IF EXISTS stocks_final;")
conn.commit()
build_final_table(conn, cur)
df_final = export_final_table(conn, cur, r'C:\Users\timur\Desktop\Python_projects_2026\stocks_final.csv')

Final table built successfully!
Exported 619040 rows to C:\Users\timur\Desktop\Python_projects_2026\stocks_final.csv


## Export

The final table is exported to a CSV file for offline analysis. `cur.description` is a psycopg2 feature that stores metadata about the last executed query — including column names — so we can build the DataFrame columns automatically without hardcoding them.

In [55]:
# check the shape
print(df_final.shape)

# check no weird nulls in key columns
print(df_final[['ticker', 'date', 'daily_return', 'log_return', 
                'rolling_std_30', 'volatility_regime', 
                'sharpe_ratio', 'max_drawdown']].isnull().sum())

# peek at a single stock
print(df_final[df_final['ticker'] == 'AAPL'].head(10))

# regime distribution - how many days in each regime across all stocks
print(df_final['volatility_regime'].value_counts())

# top 10 stocks by sharpe ratio
print(df_final.drop_duplicates('ticker')[['ticker', 'sharpe_ratio', 'annualized_return', 'max_drawdown']]
      .sort_values('sharpe_ratio', ascending=False)
      .head(10))

(619040, 35)
ticker                 0
date                   0
daily_return           0
log_return             0
rolling_std_30       505
volatility_regime    505
sharpe_ratio           0
max_drawdown           0
dtype: int64
     ticker        date     open     high      low    close     volume  \
3777   AAPL  2013-02-08  67.7142  68.4014  66.8928  67.8542  158168416   
3778   AAPL  2013-02-11  68.0714  69.2771  67.6071  68.5614  129029425   
3779   AAPL  2013-02-12  68.5014  68.9114  66.8205  66.8428  151829363   
3780   AAPL  2013-02-13  66.7442  67.6628  66.1742  66.7156  118721995   
3781   AAPL  2013-02-14  66.3599  67.3771  66.2885  66.6556   88809154   
3782   AAPL  2013-02-15  66.9785  67.1656  65.7028  65.7371   97924631   
3783   AAPL  2013-02-19  65.8714  66.1042  64.8356  65.7128  108854046   
3784   AAPL  2013-02-20  65.3842  65.3842  64.1142  64.1214  118891367   
3785   AAPL  2013-02-21  63.7142  64.1671  63.2599  63.7228  111596821   
3786   AAPL  2013-02-22  64.1785  

## Sanity Check

We verify the final table looks correct:
- **Shape:** `(619040, 35)` — 619K rows as expected, 35 columns total.
- **Nulls:** only `rolling_std_30` and `volatility_regime` have 505 nulls each. These are the first 29 rows of each ticker where there aren't enough previous days to fill a 30-day window — completely expected.
- **AAPL preview:** returns and rolling stats look correct for the first 10 rows.
- **Regime distribution:** 52.7% Normal, 33.3% Low, 12.1% High, 1.9% Extreme. The 2013–2018 period was a long bull market with no major crashes, which explains why over 85% of stock-days fall in Low or Normal regime.

In [56]:
top10 = (df_final.drop_duplicates('ticker')
         [['ticker', 'sharpe_ratio', 'annualized_return', 'max_drawdown', 'calmar_ratio']]
         .sort_values('sharpe_ratio', ascending=False)
         .head(10)
         .reset_index(drop=True))
print(top10)

  ticker  sharpe_ratio  annualized_return  max_drawdown  calmar_ratio
0    DXC      2.185990           0.619045     -0.080531      7.687032
1    HLT      2.013412           0.458517     -0.097760      4.690231
2    NOC      1.973502           0.390304     -0.109872      3.552345
3   NVDA      1.819406           0.820620     -0.252741      3.246883
4    LMT      1.788225           0.323782     -0.135009      2.398215
5    RTN      1.605658           0.313122     -0.149733      2.091203
6    STZ      1.562683           0.483604     -0.151250      3.197386
7   CTAS      1.540701           0.315097     -0.141131      2.232656
8    HII      1.524364           0.409265     -0.275653      1.484709
9     BA      1.522869           0.360896     -0.315015      1.145647


## Top 10 Stocks by Sharpe Ratio

DXC has the best risk-adjusted return at 2.19 — it earned 62% annually while keeping drawdown to only 8.1%. NVDA had the highest raw return (82% annualized) but only ranks 4th in Sharpe because it carried much higher volatility and a 25% max drawdown. This illustrates exactly why Sharpe ratio matters: raw return alone doesn't tell the full story.

In [57]:
print(df_final['volatility_regime'].value_counts(normalize=True).mul(100).round(1))

volatility_regime
Normal     52.7
Low        33.3
High       12.1
Extreme     1.9
Name: proportion, dtype: float64


## Volatility Regime Distribution

Over 86% of all stock-days across the 2013–2018 period fell in Low or Normal volatility regimes, consistent with the sustained bull market environment. Only 1.9% of days were classified as Extreme — roughly 12,000 stock-days — showing the metric does capture real volatility events even in a calm period.